# ORMAS-CI for Transition Metal Complexes

Transition metal complexes pose a major challenge for quantum chemistry. Their
electronic structure involves multiple orbital types -- metal d-orbitals, ligand
sigma bonds, pi bonds, and lone pairs -- that participate in bonding to varying
degrees. A full CASCI treatment of the entire active space is often impractical
because the number of determinants grows combinatorially.

ORMAS-CI addresses this by partitioning the active space into chemically
motivated subspaces (e.g., metal vs. ligand orbitals) and restricting
inter-subspace excitations. This captures the essential physics while
dramatically reducing the determinant count.

In this notebook, we demonstrate the multi-subspace concept using
formaldehyde (CH2O) with an 6-31G basis. Formaldehyde has a natural
partitioning into sigma, pi, and lone-pair orbital subspaces, analogous
to the metal/ligand partitioning in transition metal complexes.

In [ ]:
from pyscf import gto, scf, mcscf
from pyscf.ormas_ci import ORMASFCISolver, ORMASConfig, Subspace
from pyscf.ormas_ci.determinants import count_determinants, casci_determinant_count
import numpy as np

## Setting up the system

We use formaldehyde (CH2O) with the minimal 6-31G basis set. This molecule
is small enough to run quickly but rich enough to demonstrate meaningful
subspace partitioning:

- **Sigma subspace**: C-O sigma bonding and antibonding orbitals
- **Pi subspace**: C-O pi bonding and antibonding orbitals
- **Lone-pair subspace**: Oxygen lone-pair orbitals

This three-subspace partitioning mirrors how one would separate metal
d-orbitals from ligand sigma and pi orbitals in a transition metal complex.

In [ ]:
# Build formaldehyde and run RHF
mol = gto.M(
    atom='C 0 0 0; O 0 0 1.2; H 0 0.94 -0.54; H 0 -0.94 -0.54',
    basis='6-31g',
    verbose=0,
)
mf = scf.RHF(mol)
mf.verbose = 0
mf.run()

print(f"Molecule: CH2O (formaldehyde)")
print(f"Basis: 6-31G")
print(f"Number of AOs: {mol.nao_nr()}")
print(f"Number of electrons: {mol.nelectron}")
print(f"RHF energy: {mf.e_tot:.10f} Hartree")

## Defining metal and ligand subspaces

We define a 6-orbital, 8-electron active space and partition it into three
subspaces that reflect the bonding character of the orbitals:

| Subspace | Orbitals | Description | Electrons allowed |
|----------|----------|-------------|-------------------|
| sigma    | 0, 1     | C-O sigma bond/antibond | 1 to 4 |
| pi       | 2, 3     | C-O pi bond/antibond    | 1 to 4 |
| lone_pair| 4, 5     | O lone pairs            | 2 to 4 |

The occupation constraints restrict inter-subspace excitations. For example,
the lone-pair subspace is constrained to hold at least 2 electrons,
preventing unphysical configurations where both lone pairs are fully
depleted. Similarly, the sigma and pi subspaces each require at least
1 electron.

In [ ]:
# Active space parameters
ncas = 6
nelecas = (4, 4)

# Define three subspaces: sigma, pi, and lone-pair
config = ORMASConfig(
    subspaces=[
        Subspace("sigma", [0, 1], min_electrons=1, max_electrons=4),
        Subspace("pi", [2, 3], min_electrons=1, max_electrons=4),
        Subspace("lone_pair", [4, 5], min_electrons=2, max_electrons=4),
    ],
    n_active_orbitals=ncas,
    nelecas=nelecas,
)

# Verify the configuration is valid
config.validate()

print("ORMAS Configuration:")
print(f"  Active orbitals: {ncas}")
print(f"  Active electrons: {nelecas[0]} alpha + {nelecas[1]} beta = {sum(nelecas)} total")
print(f"  Number of subspaces: {config.n_subspaces}")
print()
for sub in config.subspaces:
    print(f"  Subspace '{sub.name}':")
    print(f"    Orbitals: {sub.orbital_indices} ({sub.n_orbitals} orbitals)")
    print(f"    Electrons: [{sub.min_electrons}, {sub.max_electrons}]")

In [ ]:
# Run ORMAS-CI
mc_ormas = mcscf.CASCI(mf, ncas, nelecas)
mc_ormas.verbose = 0
mc_ormas.fcisolver = ORMASFCISolver(config)
e_ormas = mc_ormas.kernel()[0]

print(f"ORMAS-CI energy: {e_ormas:.10f} Hartree")

## Comparison with full CASCI

To quantify the accuracy of the ORMAS restriction, we run a full CASCI
calculation with the same active space. The CASCI energy is the variational
lower bound -- any restriction on the determinant space can only raise the
energy.

In [ ]:
# Full CASCI reference
mc_cas = mcscf.CASCI(mf, ncas, nelecas)
mc_cas.verbose = 0
e_cas = mc_cas.kernel()[0]

error_hartree = e_ormas - e_cas
error_mhartree = error_hartree * 1000
error_kcal = error_hartree * 627.509

print(f"Full CASCI energy:  {e_cas:.10f} Hartree")
print(f"ORMAS-CI energy:    {e_ormas:.10f} Hartree")
print(f"Energy difference:  {error_mhartree:.4f} mHartree ({error_kcal:.4f} kcal/mol)")
print()
if abs(error_mhartree) < 1.0:
    print("The ORMAS energy is within 1 mHartree of the CASCI reference.")
    print("This is often considered 'chemical accuracy' for relative energies.")
else:
    print(f"The ORMAS restriction introduces a {error_mhartree:.2f} mHartree error.")
    print("Relaxing the occupation constraints would reduce this error.")

## Determinant space analysis

The key advantage of ORMAS-CI is the reduction in the number of determinants.
Fewer determinants means a smaller CI Hamiltonian, faster diagonalization,
and -- for quantum computing -- fewer qubits and shallower circuits.

In [ ]:
n_ormas = count_determinants(config)
n_casci = casci_determinant_count(ncas, nelecas)
reduction_pct = (1 - n_ormas / n_casci) * 100

print("Determinant Space Comparison")
print("=" * 45)
print(f"Full CASCI determinants:    {n_casci:>10,}")
print(f"ORMAS-CI determinants:      {n_ormas:>10,}")
print(f"Reduction:                  {reduction_pct:>9.1f}%")
print(f"ORMAS/CASCI ratio:          {n_ormas/n_casci:>9.2%}")
print()
print(f"The ORMAS restriction eliminates {n_casci - n_ormas} determinants")
print(f"that violate the subspace occupation constraints.")

## Dominant configurations

We can examine the CI vector to see which determinants contribute most to
the ground-state wavefunction. This tells us about the electronic structure:
which orbital occupations are most important.

For each determinant, we show its CI coefficient and the orbital occupation
pattern (2 = doubly occupied, a = alpha only, b = beta only, 0 = empty).

In [ ]:
from pyscf.ormas_ci.determinants import build_determinant_list

# Get the CI vector and determinant strings from the ORMAS calculation
ci_vec = mc_ormas.ci
alpha_strings, beta_strings = build_determinant_list(config)


def format_occupation(alpha_str, beta_str, n_orb):
    """Format a determinant as an occupation string."""
    occ = []
    for i in range(n_orb):
        a = bool(alpha_str & (1 << i))
        b = bool(beta_str & (1 << i))
        if a and b:
            occ.append("2")
        elif a:
            occ.append("a")
        elif b:
            occ.append("b")
        else:
            occ.append("0")
    return "".join(occ)


def get_subspace_occupation(alpha_str, beta_str, subspace):
    """Count total electrons in a subspace for a given determinant."""
    count = 0
    for idx in subspace.orbital_indices:
        if alpha_str & (1 << idx):
            count += 1
        if beta_str & (1 << idx):
            count += 1
    return count


# Sort by absolute CI coefficient (descending)
sorted_indices = np.argsort(np.abs(ci_vec))[::-1]

# Print subspace labels
print("Orbital subspace assignment:")
labels = []
for i in range(ncas):
    sub = config.get_subspace_for_orbital(i)
    labels.append(sub.name[0].upper())  # First letter
print(f"  Orbital:   {''.join(str(i) for i in range(ncas))}")
print(f"  Subspace:  {''.join(labels)}")
print(f"  (S=sigma, P=pi, L=lone_pair)")
print()

# Print top determinants
n_top = min(10, len(ci_vec))
print(f"Top {n_top} determinants by |coefficient|:")
print(f"{'Rank':>4}  {'Coeff':>10}  {'|Coeff|^2':>10}  {'Occupation':>10}  {'sigma':>5} {'pi':>4} {'lp':>4}")
print("-" * 60)

cumulative_weight = 0.0
for rank, idx in enumerate(sorted_indices[:n_top]):
    coeff = ci_vec[idx]
    weight = coeff ** 2
    cumulative_weight += weight
    occ = format_occupation(alpha_strings[idx], beta_strings[idx], ncas)

    sub_occs = []
    for sub in config.subspaces:
        sub_occs.append(get_subspace_occupation(
            alpha_strings[idx], beta_strings[idx], sub
        ))

    print(
        f"{rank+1:>4}  {coeff:>10.6f}  {weight:>10.6f}  "
        f"{occ:>10}  {sub_occs[0]:>5d} {sub_occs[1]:>4d} {sub_occs[2]:>4d}"
    )

print(f"\nCumulative weight of top {n_top} determinants: {cumulative_weight:.4f}")

## Quantum computing implications

For quantum computing applications, the subspace factorization has direct
implications for resource requirements:

- **Qubit count**: Under Jordan-Wigner encoding, each spatial orbital requires
  2 qubits (one for alpha, one for beta). If the subspaces can be treated
  independently or with limited coupling, the qubit requirement is determined
  by the largest subspace rather than the full active space.

- **Circuit depth**: Fewer allowed excitations means fewer gates in a
  VQE ansatz or shorter Trotter steps in quantum phase estimation.

- **Determinant count**: The reduced determinant space translates to fewer
  terms in the Hamiltonian and fewer parameters in the quantum circuit.

In [ ]:
# Qubit resource estimation
full_qubits = 2 * ncas  # Jordan-Wigner: 2 qubits per spatial orbital
max_subspace_orbitals = max(sub.n_orbitals for sub in config.subspaces)
subspace_qubits = 2 * max_subspace_orbitals
qubit_savings_pct = (1 - subspace_qubits / full_qubits) * 100

print("Quantum Resource Estimation (Jordan-Wigner encoding)")
print("=" * 55)
print()
print("Qubit requirements:")
print(f"  Full CASCI approach:       {full_qubits} qubits ({ncas} spatial orbitals)")
print(f"  Largest ORMAS subspace:    {subspace_qubits} qubits ({max_subspace_orbitals} spatial orbitals)")
print(f"  Qubit reduction:           {qubit_savings_pct:.0f}%")
print()
print("Per-subspace breakdown:")
for sub in config.subspaces:
    q = 2 * sub.n_orbitals
    print(f"  {sub.name:>12}: {sub.n_orbitals} orbitals -> {q} qubits")
print()
print("Determinant space:")
print(f"  CASCI:  {n_casci:>8,} determinants")
print(f"  ORMAS:  {n_ormas:>8,} determinants")
print(f"  Ratio:  {n_ormas/n_casci:.2%}")
print()
print("When subspaces can be treated on separate quantum registers,")
print(f"the qubit count drops from {full_qubits} to {subspace_qubits}.")
print("For larger active spaces typical of transition metal complexes")
print("(e.g., 10-20 active orbitals), these savings become substantial.")

## Scaling to larger systems

To illustrate how the savings grow with system size, we estimate
determinant counts for hypothetical active spaces representative of
transition metal complexes, without running the full calculations.

In [ ]:
# Project savings to larger active spaces
print("Projected savings for transition-metal-scale active spaces")
print("=" * 65)
print(f"{'ncas':>6} {'nelecas':>12} {'CASCI dets':>14} {'ORMAS dets':>14} {'Ratio':>8}")
print("-" * 65)

test_cases = [
    # (ncas, nelecas, metal_orbs, ligand_orbs, metal_elec_range, ligand_elec_range)
    (6, (3, 3), [0, 1, 2], [3, 4, 5], (2, 4), (2, 4)),
    (8, (4, 4), [0, 1, 2, 3], [4, 5, 6, 7], (3, 5), (3, 5)),
    (10, (5, 5), [0, 1, 2, 3, 4], [5, 6, 7, 8, 9], (4, 6), (4, 6)),
    (12, (6, 6), [0, 1, 2, 3, 4, 5], [6, 7, 8, 9, 10, 11], (5, 7), (5, 7)),
    (14, (7, 7), list(range(7)), list(range(7, 14)), (6, 8), (6, 8)),
]

for ncas_t, nelecas_t, metal, ligand, m_range, l_range in test_cases:
    cfg = ORMASConfig(
        subspaces=[
            Subspace("metal", metal, min_electrons=m_range[0], max_electrons=m_range[1]),
            Subspace("ligand", ligand, min_electrons=l_range[0], max_electrons=l_range[1]),
        ],
        n_active_orbitals=ncas_t,
        nelecas=nelecas_t,
    )
    n_o = count_determinants(cfg)
    n_c = casci_determinant_count(ncas_t, nelecas_t)
    print(f"{ncas_t:>6} {str(nelecas_t):>12} {n_c:>14,} {n_o:>14,} {n_o/n_c:>7.2%}")

print()
print("As the active space grows, the ORMAS reduction becomes more dramatic.")
print("For a (14,14) active space, ORMAS reduces the determinant count")
print("by a large factor, enabling calculations that would be infeasible")
print("with full CASCI on near-term quantum hardware.")

## Summary

This notebook demonstrated the core ORMAS-CI workflow for systems with
multiple orbital types, using formaldehyde as a proxy for transition metal
complexes:

1. **Subspace definition** -- The active space was partitioned into sigma,
   pi, and lone-pair subspaces with occupation constraints that enforce
   chemically reasonable electron distributions.

2. **Accuracy** -- The ORMAS-CI energy was compared to the full CASCI
   reference, showing the tradeoff between accuracy and computational
   savings.

3. **Determinant reduction** -- The restricted CI space is significantly
   smaller than the full CASCI space, and the savings grow with active
   space size.

4. **Quantum computing** -- Subspace factorization reduces qubit
   requirements from `2 * ncas` to `2 * max_subspace_size`, a significant
   saving for transition metal complexes where 10-20 active orbitals
   are common.

For real transition metal systems, the same workflow applies: use AVAS or
domain knowledge to identify metal d-orbitals and ligand orbitals, assign
them to separate ORMAS subspaces, and constrain the inter-subspace
excitations based on the expected electronic structure.